# Convert DOCX to PDF

Batch-converts every `.docx` under `data/mcq/Paket soal/` to PDF via LibreOffice's headless
CLI (`soffice --headless --convert-to pdf`), so they can be fed through the same ingestion
pipeline as the lecture-slide PDFs (`scripts/ingest_data.py` / `v1_pipeline.ipynb`).

LibreOffice is used rather than `docx2pdf` because `docx2pdf` on macOS just automates
Microsoft Word via AppleScript — it requires Word to be installed and doesn't work headless
in a server/CI context. `soffice --headless` has no such dependency.

Output PDFs land flat in `data/mcq/pdf/`, named after their source `.docx` (batch/folder
structure is not preserved in the filename — see the "Handle name collisions" cell below for
what happens if two docx files share a filename across different batches).

In [1]:
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT / "src"))

SOFFICE_BIN = "soffice"
result = subprocess.run([SOFFICE_BIN, "--version"], capture_output=True, text=True)
assert result.returncode == 0, (
    "soffice not found on PATH. Install LibreOffice first:\n"
    "  brew install --cask libreoffice"
)
print(result.stdout.strip())

LibreOffice 26.2.5.2 cd7284b4cbbfeb507e630c1aac019f4157393acb


## Find source docx files

In [2]:
DOCX_SOURCE_DIR = PROJECT_ROOT / "data" / "mcq" / "Paket soal"
PDF_OUTPUT_DIR = PROJECT_ROOT / "data" / "mcq" / "pdf"
PDF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

docx_paths = sorted(DOCX_SOURCE_DIR.rglob("*.docx"))
print(f"found {len(docx_paths)} docx file(s) under {DOCX_SOURCE_DIR}")

found 264 docx file(s) under /Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student/data/mcq/Paket soal


## Check for filename collisions

Output is flat (`data/mcq/pdf/{stem}.pdf`), but source files are nested under per-batch
folders (`Batch 1/`, `Batch 2/`, ...) and could share a filename across batches (e.g. two
different `Paket 1.docx`). Collisions would silently overwrite one converted PDF with
another, so this is checked explicitly before converting anything.

In [3]:
from collections import defaultdict

by_stem = defaultdict(list)
for p in docx_paths:
    by_stem[p.stem].append(p)

collisions = {stem: paths for stem, paths in by_stem.items() if len(paths) > 1}
if collisions:
    print(f"{len(collisions)} filename collision(s) found:")
    for stem, paths in collisions.items():
        print(f"  {stem}:")
        for p in paths:
            print(f"    {p.relative_to(PROJECT_ROOT)}")
else:
    print("no filename collisions")

34 filename collision(s) found:
  Paket 1:
    data/mcq/Paket soal/Batch 1/Paket 1.docx
    data/mcq/Paket soal/Batch 1/Paket Day 2/Paket 1.docx
  Paket 10:
    data/mcq/Paket soal/Batch 1/Paket 10.docx
    data/mcq/Paket soal/Batch 1/Paket Day 2/Paket 10.docx
  Paket 2:
    data/mcq/Paket soal/Batch 1/Paket 2.docx
    data/mcq/Paket soal/Batch 1/Paket Day 2/Paket 2.docx
  Paket 21:
    data/mcq/Paket soal/Batch 1/Paket 21.docx
    data/mcq/Paket soal/Batch 1/Paket Day 2/Paket 21.docx
  Paket 22:
    data/mcq/Paket soal/Batch 1/Paket 22.docx
    data/mcq/Paket soal/Batch 1/Paket Day 2/Paket 22.docx
  Paket 23:
    data/mcq/Paket soal/Batch 1/Paket 23.docx
    data/mcq/Paket soal/Batch 1/Paket Day 2/Paket 23.docx
  Paket 24:
    data/mcq/Paket soal/Batch 1/Paket 24.docx
    data/mcq/Paket soal/Batch 1/Paket Day 2/Paket 24.docx
  Paket 25:
    data/mcq/Paket soal/Batch 1/Paket 25.docx
    data/mcq/Paket soal/Batch 1/Paket Day 2/Paket 25.docx
  Paket 26:
    data/mcq/Paket soal/Batch 1/Pa

If collisions were found above, resolve them before continuing (e.g. rename source files
to include their batch, or convert into per-batch output subfolders instead of one flat
directory) — the conversion cell below does not guard against overwrites.

## Convert

Each file is converted independently via a fresh `soffice --headless` invocation (safer
than the `--convert-to pdf *.docx` batch form, which can hang the whole batch if one file
triggers LibreOffice's macro/format-recovery dialog). Failures are caught per-file and
logged, matching the rest of this repo's fail-soft-per-item convention (see
`scripts/ingest_data.py`) — one bad docx shouldn't abort the whole run.

In [4]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("docx_to_pdf")

CONVERT_TIMEOUT_SECONDS = 120


def convert_one(docx_path: Path, output_dir: Path) -> Path | None:
    cmd = [
        SOFFICE_BIN,
        "--headless",
        "--convert-to", "pdf",
        "--outdir", str(output_dir),
        str(docx_path),
    ]
    try:
        result = subprocess.run(
            cmd, capture_output=True, text=True, timeout=CONVERT_TIMEOUT_SECONDS
        )
    except subprocess.TimeoutExpired:
        log.warning("timed out converting %s", docx_path.name)
        return None

    if result.returncode != 0:
        log.warning("soffice failed on %s: %s", docx_path.name, result.stderr.strip()[-500:])
        return None

    expected = output_dir / f"{docx_path.stem}.pdf"
    if not expected.exists():
        log.warning("soffice reported success but no output found for %s", docx_path.name)
        return None
    return expected

In [5]:
from tqdm.auto import tqdm

converted: list[Path] = []
failures: list[Path] = []

for docx_path in tqdm(docx_paths, desc="converting"):
    pdf_path = convert_one(docx_path, PDF_OUTPUT_DIR)
    if pdf_path is None:
        failures.append(docx_path)
    else:
        converted.append(pdf_path)

print(f"converted {len(converted)}/{len(docx_paths)}, {len(failures)} failure(s)")

/Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
converting: 100%|██████████| 264/264 [04:26<00:00,  1.01s/it]

converted 264/264, 0 failure(s)


In [6]:
if failures:
    print("failed conversions:")
    for f in failures:
        print(f"  {f.relative_to(PROJECT_ROOT)}")
else:
    print("no failures")

no failures


## Next step

Converted PDFs are now under `data/mcq/pdf/`, ready for the same ingestion path as the
lecture-slide PDFs:

```
.venv/bin/python scripts/ingest_data.py --dir data/mcq/pdf
```

See `docs/03-ingestion-pipeline.md` for what that does.